In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — LECTEUR DE NUMÉROS v2 : entraînement sur TOUS les matchs, deux équipes (06/11/2026). Mode audit.
# PARSeq (baudm/parseq, Apache 2.0) FIGÉ au commit 1902db043c029a7e03a3818c616c06600af574be.
# Étiquettes, par match (Minerois Ster EXCLU, réservé à l'évaluation) :
#   1. le modèle v1 (parseq_scoutia_maillots_v1.pt) lit tous les morceaux de suivi, deux équipes, gratuitement ;
#   2. CAS SÛRS étiquetés par v1 : même numéro sur ≥ 2 images à confiance ≥ 0,90, aucun autre numéro ≥ 0,50 ;
#   3. CAS DOUTEUX tranchés par Gemini (4 vignettes du joueur), au plus MAX_GEMINI questions par match ;
#   + les étiquettes Gemini de Franchimont (lectures_v27_franchimont.json).
# Lecture restreinte aux CHIFFRES 0-9 (lettres interdites au décodage). Apprentissage repart du PARSeq d'origine.
# Évaluation sur Minerois Ster (jamais vu) : modèle d'origine / v1 / v2, accord avec Gemini, segments du n°31.
# Planches par numéro (planche_etiquettes_<n>.jpg) pour vérifier les étiquettes à l'œil.
# Coût Gemini ≈ 0,45 $ par match au plus (plafond global 4,50 $). GPU. Durée : plusieurs heures (reprise possible :
# les étiquettes de chaque match sont enregistrées dans etiquettes_<match>.pkl et réutilisées si présentes).
# Entrées : vidéos des matchs, codeprojectai (caches), mziparameter, parseq_scoutia_maillots_v1.pt,
# lectures_v27_franchimont.json, lectures_v28_minerois_ster.json. Internet activé.
# ════════════════════════════════════════════════════════════════════════════════════════════
V = "/kaggle/input/datasets/michaelziant"
CODE = f"{V}/codeprojectai"
MATCHS = {   # match : (vidéo, cache de suivi)
    "franchimont": (f"{V}/franchimont-complet/ster-b-franchimont-b-2026-04-09.mp4", f"{CODE}/cache_franchimont_reference"),
    "raeren": (f"{V}/raeren-complet/ster-b-raeren-b-2026-04-03.mp4", f"{CODE}/cache_raeren_reference"),
    "spa": (f"{V}/spamatchp4/match-p4-2026-08-29.mp4", f"{CODE}/cache_spa_reference"),
    "goe": (f"{V}/goe-complet/ster-b-goe-2026-03-20.mp4", f"{CODE}/cache_goe_reference"),
    "p1_minerois": (f"{V}/p1minerois-complet/p1-minerois-2026-03-28.mp4", f"{CODE}/cache_p1_minerois_reference"),
    "stembert": (f"{V}/stembert-complet/ster-b-stembert-2026-03-07.mp4", f"{CODE}/cache_stembert_reference"),
    "wanze": (f"{V}/wanze-complet/ster-wanze-bas-oha-2026-02-28.mp4", f"{CODE}/cache_wanze_reference"),
    "juprelle": (f"{V}/juprelle-complet/ster-juprelle-2026-01-25.mp4", f"{CODE}/cache_juprelle_reference"),
}
# 06/11 : ESSAI INTERMÉDIAIRE — seuls ces matchs sont traités (les autres ne sont ni lus ni étiquetés) ;
# MATCH_TEST est exclu de l'apprentissage et sert de 2e évaluation (autres tenues que le rouge).
MATCHS_ACTIFS = ["franchimont", "raeren", "spa", "goe"]          # [] = tous
MATCH_TEST = "spa"                                               # "" = aucun
EVAL = ("minerois_ster", f"{V}/mineroisster-complet/minerois-ster-2025-11-09.mp4", f"{CODE}/cache_minerois_ster_reference")
NUMERO_CONTROLE = "31"
COMMIT = "1902db043c029a7e03a3818c616c06600af574be"
MAX_SEGMENTS = 3000      # par match (tirés au hasard parmi les morceaux exploitables)
MIN_DUREE_S, MIN_HAUTEUR = 2.0, 50
MAX_GEMINI, PLAFOND = 150, 9.00   # 06/11 : ≈ 1 $ par match mesuré (planches plus grandes que prévu)
EPOQUES, LOT, LR = 10, 64, 5e-5
SORTIE = "/kaggle/working"
MODELE_GEMINI = "gemini-3.8-flash"

import os, sys, glob, json, pickle, time, random, collections, subprocess
import numpy as np
import cv2
import torch
T0 = time.time(); random.seed(0); torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "pytorch-lightning", "nltk"], check=False)
from torchvision import transforms as T


def _f(nom):
    return next(iter(sorted(glob.glob(f"/kaggle/input/**/{nom}", recursive=True)) + glob.glob(f"{SORTIE}/{nom}")), None)


def nouveau_parseq():
    return torch.hub.load(f"baudm/parseq:{COMMIT}", "parseq", pretrained=True, trust_repo=True).to(dev)


V1 = _f("parseq_scoutia_maillots_v1.pt")
if not V1:
    raise SystemExit("❌ parseq_scoutia_maillots_v1.pt absent des entrées")
modele = nouveau_parseq(); modele.load_state_dict(torch.load(V1, map_location="cpu")); modele.eval()
_itos = getattr(modele.tokenizer, "_itos", None)
MASQUE = None
if _itos:
    MASQUE = torch.tensor([0.0 if (i == 0 or str(c).isdigit()) else float("-inf") for i, c in enumerate(_itos)], device=dev)
    print(f"   lecture restreinte aux chiffres : {sum(str(c).isdigit() for c in _itos)} chiffres autorisés + fin de texte")
_norm = T.Compose([T.ToPILImage(), T.Resize((32, 128), T.InterpolationMode.BICUBIC), T.ToTensor(), T.Normalize(0.5, 0.5)])
_aug = T.Compose([T.ToPILImage(), T.ColorJitter(0.4, 0.4, 0.3, 0.05), T.RandomAffine(4, translate=(0.04, 0.06), scale=(0.9, 1.1)),
                  T.Resize((32, 128), T.InterpolationMode.BICUBIC), T.ToTensor(), T.Normalize(0.5, 0.5)])
rgb = lambda c: cv2.cvtColor(c, cv2.COLOR_BGR2RGB)
dejpg = lambda b: cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_COLOR)
enjpg = lambda c: cv2.imencode(".jpg", c, [cv2.IMWRITE_JPEG_QUALITY, 92])[1].tobytes()


def chiffres(s):
    s = "".join(ch for ch in str(s) if ch.isdigit())
    return s if 1 <= len(s) <= 2 else None


def lire(m, decs):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(decs), 256):
            x = torch.stack([_norm(rgb(c)) for c in decs[i:i + 256]]).to(dev)
            lg = m(x)
            if MASQUE is not None and lg.shape[-1] == MASQUE.shape[0]:
                lg = lg + MASQUE
            lab, conf = m.tokenizer.decode(lg.softmax(-1))
            out += [(chiffres(l), float(c.prod())) for l, c in zip(lab, conf)]
    return out


def decoupes(im, b):
    x1, y1, x2, y2 = [int(v) for v in b]; h, w = y2 - y1, x2 - x1
    if h < 20 or w < 8:
        return []
    H, W = im.shape[:2]; out = []
    for a, z in ((0.12, 0.55), (0.18, 0.50)):
        c = im[max(0, y1 + int(a * h)):min(H, y1 + int(z * h)), max(0, x1 - int(0.1 * w)):min(W, x2 + int(0.1 * w))]
        if c.size:
            k = 64 / max(1, c.shape[0])
            out.append(cv2.resize(c, (max(16, int(c.shape[1] * k)), 64), interpolation=cv2.INTER_CUBIC))
    return out


def charger_suivi(cache):
    meta = json.load(open(f"{cache}/meta.json", encoding="utf-8"))
    fps, ko = float(meta.get("fps") or 29.97), float(meta.get("kickoff_offset") or 0.0)
    pistes = collections.defaultdict(list)
    for f in pickle.load(open(f"{cache}/frames_data.pkl", "rb")):
        n = int(f.get("frame", 0))
        for d in f.get("players") or []:
            tid = d.get("tracker_id") if d.get("tracker_id") is not None else d.get("track_id")
            if d.get("bbox") is not None and tid is not None:
                pistes[int(tid)].append((n, [float(v) for v in d["bbox"]]))
    return fps, ko, pistes


def segments_de(fps, pistes):
    segs = []
    for tid, pos in pistes.items():
        pos.sort(); cour = [pos[0]]
        for p in pos[1:]:
            if p[0] - cour[-1][0] > fps:                   # trou de plus d'1 s : nouveau morceau
                segs.append((tid, cour)); cour = []
            cour.append(p)
        segs.append((tid, cour))
    bons = [(tid, c) for tid, c in segs if (c[-1][0] - c[0][0]) / fps >= MIN_DUREE_S
            and np.median([b[3] - b[1] for _, b in c]) >= MIN_HAUTEUR]
    random.shuffle(bons)
    return bons[:MAX_SEGMENTS]


def extraire(video, fps, ko, segs):
    """→ {sid: [(n, [découpes dos], vignette entière)]} pour 4 images par morceau"""
    taches = []
    for tid, c in segs:
        sid = f"{tid}@{c[0][0]}-{c[-1][0]}"
        for p in [c[int(i * (len(c) - 1) / 3)] for i in range(4)]:
            taches.append((p[0], sid, p[1]))
    taches.sort(key=lambda x: x[0])
    cap = cv2.VideoCapture(video); out = collections.defaultdict(list); der = None; im = None
    for n, sid, b in taches:
        if n != der:
            cap.set(cv2.CAP_PROP_POS_MSEC, (ko + n / fps) * 1000); ok, im = cap.read(); der = n
            im = im if ok else None
        if im is None:
            continue
        x1, y1, x2, y2 = [int(v) for v in b]
        v = im[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        if v.size == 0:
            continue
        v = cv2.resize(v, (max(8, int(v.shape[1] * 200 / max(1, v.shape[0]))), 200))
        out[sid].append((n, [enjpg(d) for d in decoupes(im, b)], enjpg(v)))
    cap.release()
    return out


# ── Gemini pour les cas douteux ──
def _cle():
    v = os.getenv("GEMINI_API_KEY", "")
    pe = next(iter(glob.glob("/kaggle/input/**/mziparameter/**/param.env", recursive=True)), None)
    if not v and pe:
        for l in open(pe, encoding="utf-8"):
            if l.strip().startswith("GEMINI_API_KEY="):
                v = l.split("=", 1)[1].strip().strip('"').strip("'")
    return v


jet = {"q": 0, "e": 0, "s": 0}
dollars = lambda: jet["e"] / 1e6 * 0.75 + jet["s"] / 1e6 * 3.75
client = None
if _cle():
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=_cle())
else:
    print("   ⚠️ pas de clé Gemini : seuls les cas sûrs seront étiquetés")
QUESTION = """These 4 images show (normally) the same amateur football player at 4 moments. For EACH image, give the shirt
number ONLY if its digits are clearly readable (on the back, front or shorts); otherwise null. Do not guess.
Answer ONLY with JSON: {"numbers": [n1, n2, n3, n4]}"""


def gemini_numeros(vignettes):
    if client is None or dollars() > PLAFOND - 0.05:
        return None
    h = 200; ims = [dejpg(v) for v in vignettes]
    planche = cv2.hconcat([cv2.resize(i, (max(8, int(i.shape[1] * h / i.shape[0])), h)) for i in ims])
    try:
        r = client.models.generate_content(model=MODELE_GEMINI, contents=[QUESTION, types.Part.from_bytes(
            data=enjpg(planche), mime_type="image/jpeg")], config={"temperature": 0.0, "thinking_config": {"thinking_level": "low"}})
        um = getattr(r, "usage_metadata", None)
        jet["q"] += 1; jet["e"] += int(getattr(um, "prompt_token_count", 0) or 0)
        jet["s"] += int(getattr(um, "candidates_token_count", 0) or 0) + int(getattr(um, "thoughts_token_count", 0) or 0)
        txt = r.text or ""; d, _ = json.JSONDecoder().raw_decode(txt[txt.index("{"):])
        return [chiffres(x) if x is not None else None for x in (d.get("numbers") or [])][:len(vignettes)]
    except Exception as e:
        print(f"   ⚠️ Gemini : {e}"); return None


# ── étiquetage match par match (reprise possible) ──
ETIQ = []                        # (match, sid, jpeg découpe, étiquette, source)
for match, (video, cache) in MATCHS.items():
    if MATCHS_ACTIFS and match not in MATCHS_ACTIFS:
        continue
    fic = f"{SORTIE}/etiquettes_{match}.pkl"
    deja = _f(f"etiquettes_{match}.pkl")
    if deja:
        ETIQ += pickle.load(open(deja, "rb")); print(f"   {match} : étiquettes reprises ({deja})"); continue
    if not (os.path.exists(video) and os.path.exists(f"{cache}/frames_data.pkl")):
        print(f"   ⚠️ {match} : vidéo ou cache absent des entrées, match ignoré"); continue
    t = time.time()
    fps, ko, pistes = charger_suivi(cache)
    segs = segments_de(fps, pistes)
    data = extraire(video, fps, ko, segs)
    plat = [(sid, i, j, dj) for sid, lst in data.items() for i, (n, ds, v) in enumerate(lst) for j, dj in enumerate(ds)]
    lus = lire(modele, [dejpg(d) for _, _, _, d in plat])
    par = collections.defaultdict(dict)            # sid → image → (numéro, conf) meilleure découpe
    for (sid, i, j, dj), (num, cf) in zip(plat, lus):
        if num and (i not in par[sid] or cf > par[sid][i][1]):
            par[sid][i] = (num, cf, dj)
    etiq = []; douteux = []; n_surs = 0
    for sid, lst in data.items():
        lec = par.get(sid, {})
        forts = collections.Counter(v[0] for v in lec.values() if v[1] >= 0.90)
        moyens = {v[0] for v in lec.values() if v[1] >= 0.50}
        if forts and forts.most_common(1)[0][1] >= 2 and len(moyens) == 1:
            num = forts.most_common(1)[0][0]; n_surs += 1
            etiq += [(match, sid, v[2], num, "v1") for v in lec.values() if v[0] == num and v[1] >= 0.50]
        elif moyens:
            douteux.append((max(v[1] for v in lec.values()), sid))
    douteux.sort(reverse=True); n_g = 0
    for _, sid in douteux[:MAX_GEMINI]:
        lst = data[sid]; rep = gemini_numeros([v for _, _, v in lst])
        if not rep:
            continue
        n_g += 1
        for (n, ds, v), num in zip(lst, rep):
            if num:
                etiq += [(match, sid, d, num, "gemini") for d in ds]
    pickle.dump(etiq, open(fic, "wb"))
    ETIQ += etiq
    print(f"   {match} : {len(segs)} morceaux lus | sûrs (v1) : {n_surs} | douteux : {len(douteux)} dont {n_g} tranchés par Gemini | "
          f"{len(etiq)} découpes étiquetées | {(time.time() - t) / 60:.1f} min | Gemini cumulé ≈ {dollars():.2f} $")

# + Franchimont, étiquettes Gemini du montage joueur (comme pour v1)
lv27 = _f("lectures_v27_franchimont.json")
if lv27 and os.path.exists(MATCHS["franchimont"][0]):
    lect = json.load(open(lv27, encoding="utf-8"))["segments"]
    fps, ko, pistes = charger_suivi(MATCHS["franchimont"][1])
    idx = {(n, tid): b for tid, pos in pistes.items() for n, b in pos}
    cap = cv2.VideoCapture(MATCHS["franchimont"][0]); n27 = 0
    for sid, e in sorted(lect.items()):
        tid = int(sid.split("@")[0]); nums = e.get("numeros") or []
        for i, n in enumerate(e.get("images") or []):
            lab = chiffres(nums[i]) if i < len(nums) and nums[i] is not None else None
            if lab and (int(n), tid) in idx:
                cap.set(cv2.CAP_PROP_POS_MSEC, (ko + int(n) / fps) * 1000); ok, im = cap.read()
                if ok:
                    for d in decoupes(im, idx[(int(n), tid)]):
                        ETIQ.append(("franchimont", "v27:" + sid, enjpg(d), lab, "gemini_v27")); n27 += 1
    cap.release(); print(f"   franchimont : + {n27} découpes des lectures Gemini du montage joueur")

print(f"\n   TOTAL : {len(ETIQ)} découpes étiquetées | par source : {dict(collections.Counter(e[4] for e in ETIQ))}")
print(f"   numéros les plus fréquents : {collections.Counter(e[3] for e in ETIQ).most_common(15)}")

# planches par numéro (vérification à l'œil)
par_num = collections.defaultdict(list)
for e in ETIQ:
    par_num[e[3]].append(e)
for num, lst in sorted(par_num.items(), key=lambda x: -len(x[1]))[:12]:
    ech = random.sample(lst, min(40, len(lst))); ims = [cv2.resize(dejpg(e[2]), (96, 64)) for e in ech]
    while len(ims) % 8:
        ims.append(np.zeros((64, 96, 3), np.uint8))
    cv2.imwrite(f"{SORTIE}/planche_etiquettes_{num}.jpg", cv2.vconcat([cv2.hconcat(ims[i:i + 8]) for i in range(0, len(ims), 8)]))
print(f"   planches : {SORTIE}/planche_etiquettes_<numéro>.jpg (12 numéros les plus fréquents, 40 vignettes) — un intrus ?")

# ── évaluation Minerois Ster (jamais vu) ──
lv28 = _f("lectures_v28_minerois_ster.json")
lect_e = json.load(open(lv28, encoding="utf-8"))["segments"] if lv28 else {}
fps, ko, pistes = charger_suivi(EVAL[2]); idx = {(n, tid): b for tid, pos in pistes.items() for n, b in pos}
vign_e = []; cap = cv2.VideoCapture(EVAL[1])
for sid, e in sorted(lect_e.items(), key=lambda x: min(x[1].get("images") or [0])):
    tid = int(sid.split("@")[0])
    for n in e.get("images") or []:
        if (int(n), tid) in idx:
            cap.set(cv2.CAP_PROP_POS_MSEC, (ko + int(n) / fps) * 1000); ok, im = cap.read()
            if ok:
                vign_e += [(sid, int(n), d) for d in decoupes(im, idx[(int(n), tid)])]
cap.release()


def evaluer(m, titre):
    lus = lire(m, [v[2] for v in vign_e]); par_seg = collections.defaultdict(list)
    for (sid, n, _), (num, cf) in zip(vign_e, lus):
        par_seg[sid].append((n, num, cf))
    print(f"\n   {titre}\n   seuil | segments lus | accord Gemini (même / différent) | n°{NUMERO_CONTROLE} (commun / seul)")
    for sl in (0.9, 0.8, 0.6):
        lus_s = meme = diff = c_c = c_s = 0
        for sid, e in lect_e.items():
            par_img = {}
            for n, num, cf in par_seg.get(sid, []):
                if num and cf >= sl and (n not in par_img or cf > par_img[n][1]):
                    par_img[n] = (num, cf)
            c = collections.Counter(v[0] for v in par_img.values())
            np_ = c.most_common(1)[0][0] if c and c.most_common(1)[0][1] >= 2 else None
            g = [x for x in (e.get("numeros") or []) if x is not None]
            ng = collections.Counter(map(str, g)).most_common(1)[0][0] if g else None
            lus_s += bool(np_)
            if np_ and ng:
                meme += np_ == ng; diff += np_ != ng
            if np_ == NUMERO_CONTROLE:
                c_c += ng == NUMERO_CONTROLE; c_s += ng != NUMERO_CONTROLE
        print(f"   {sl:5.2f} | {lus_s:12d} | {meme:4d} / {diff:<4d}                     | {c_c} / {c_s}")


if vign_e:
    evaluer(nouveau_parseq(), "MODÈLE D'ORIGINE (chiffres seuls) — Minerois Ster")
    evaluer(modele, "v1 (Franchimont seul, chiffres seuls) — Minerois Ster")

# ── 2e évaluation : le match de test, exclu de l'apprentissage (vignette par vignette, étiquettes Gemini/v1) ──
TEST = [e for e in ETIQ if e[0] == MATCH_TEST]
ETIQ = [e for e in ETIQ if e[0] != MATCH_TEST]


def evaluer_test(m, titre):
    if not TEST:
        return
    lus = lire(m, [dejpg(e[2]) for e in TEST])
    print(f"\n   {titre} — {MATCH_TEST} ({len(TEST)} découpes, exclu de l'apprentissage)")
    print("   seuil | découpes lues | justes | fausses | justes parmi celles de Gemini")
    for sl in (0.9, 0.8, 0.6):
        lu = [(n, e) for (n, cf), e in zip(lus, TEST) if n and cf >= sl]
        ok = sum(n == e[3] for n, e in lu); ok_g = sum(n == e[3] for n, e in lu if e[4] == "gemini")
        n_g = sum(1 for _, e in lu if e[4] == "gemini")
        print(f"   {sl:5.2f} | {len(lu):13d} | {ok:6d} | {len(lu) - ok:7d} | {ok_g}/{n_g}")


evaluer_test(modele, "v1")

# ── apprentissage v2 (repart du PARSeq d'origine) ──
segs_all = sorted({(e[0], e[1]) for e in ETIQ}); random.shuffle(segs_all)
val = set(segs_all[: max(1, len(segs_all) // 10)])
train = [(e[2], e[3]) for e in ETIQ if (e[0], e[1]) not in val]
vd = [(e[2], e[3]) for e in ETIQ if (e[0], e[1]) in val]
m2 = nouveau_parseq(); m2.log = lambda *a, **k: None
opt = torch.optim.AdamW(m2.parameters(), lr=LR, weight_decay=0.01)
print(f"\n   apprentissage v2 : {len(train)} découpes | validation : {len(vd)}")
for ep in range(EPOQUES):
    m2.train(); random.shuffle(train); pertes = []
    for i in range(0, len(train), LOT):
        b = train[i:i + LOT]
        x = torch.stack([_aug(rgb(dejpg(c))) for c, _ in b]).to(dev)
        p = m2.training_step((x, [l for _, l in b]), i); p = p["loss"] if isinstance(p, dict) else p
        opt.zero_grad(); p.backward(); torch.nn.utils.clip_grad_norm_(m2.parameters(), 20); opt.step(); pertes.append(float(p))
    pr = lire(m2, [dejpg(c) for c, _ in vd])
    print(f"   époque {ep + 1:2d} | perte {np.mean(pertes):.3f} | validation : {np.mean([n == l for (n, _), (_, l) in zip(pr, vd)]):.1%}")
torch.save(m2.state_dict(), f"{SORTIE}/parseq_scoutia_maillots_v2.pt")
if vign_e:
    evaluer(m2, "v2 (chiffres seuls) — Minerois Ster")
evaluer_test(m2, "v2")
print(f"\n   poids : {SORTIE}/parseq_scoutia_maillots_v2.pt (PARSeq © Darwin Bautista, Apache 2.0, commit {COMMIT[:7]})")
print(f"   Gemini : {jet['q']} question(s) ≈ {dollars():.2f} $ | durée totale : {(time.time() - T0) / 60:.0f} min")

Downloading: "https://github.com/baudm/parseq/zipball/1902db043c029a7e03a3818c616c06600af574be" to /root/.cache/torch/hub/1902db043c029a7e03a3818c616c06600af574be.zip


/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Downloading: "https://github.com/baudm/parseq/releases/download/v1.0.0/parseq-bb5792a6.pt" to /root/.cache/torch/hub/checkpoints/parseq-bb5792a6.pt


100%|██████████| 91.0M/91.0M [00:00<00:00, 188MB/s]


   lecture restreinte aux chiffres : 10 chiffres autorisés + fin de texte
   franchimont : étiquettes reprises (/kaggle/input/datasets/michaelziant/codeprojectai/etiquettes_franchimont.pkl)
   raeren : étiquettes reprises (/kaggle/input/datasets/michaelziant/codeprojectai/etiquettes_raeren.pkl)
   spa : étiquettes reprises (/kaggle/input/datasets/michaelziant/codeprojectai/etiquettes_spa.pkl)
   goe : étiquettes reprises (/kaggle/input/datasets/michaelziant/codeprojectai/etiquettes_goe.pkl)
   franchimont : + 4638 découpes des lectures Gemini du montage joueur

   TOTAL : 6964 découpes étiquetées | par source : {'v1': 908, 'gemini': 1418, 'gemini_v27': 4638}
   numéros les plus fréquents : [('10', 982), ('3', 902), ('8', 549), ('5', 540), ('26', 478), ('17', 472), ('11', 452), ('41', 398), ('7', 391), ('13', 390), ('30', 309), ('9', 252), ('6', 164), ('18', 149), ('14', 111)]
   planches : /kaggle/working/planche_etiquettes_<numéro>.jpg (12 numéros les plus fréquents, 40 vignettes) — u

Using cache found in /root/.cache/torch/hub/baudm_parseq_1902db043c029a7e03a3818c616c06600af574be



   MODÈLE D'ORIGINE (chiffres seuls) — Minerois Ster
   seuil | segments lus | accord Gemini (même / différent) | n°31 (commun / seul)
    0.90 |           10 |    8 / 0                        | 1 / 0
    0.80 |           16 |   12 / 1                        | 2 / 0
    0.60 |           27 |   19 / 2                        | 4 / 0

   v1 (Franchimont seul, chiffres seuls) — Minerois Ster
   seuil | segments lus | accord Gemini (même / différent) | n°31 (commun / seul)
    0.90 |           81 |   66 / 3                        | 11 / 1
    0.80 |           89 |   72 / 3                        | 12 / 1
    0.60 |          119 |   75 / 11                       | 12 / 1

   v1 — spa (365 découpes, exclu de l'apprentissage)
   seuil | découpes lues | justes | fausses | justes parmi celles de Gemini
    0.90 |           354 |    354 |       0 | 299/299
    0.80 |           354 |    354 |       0 | 299/299
    0.60 |           357 |    354 |       3 | 299/302


Using cache found in /root/.cache/torch/hub/baudm_parseq_1902db043c029a7e03a3818c616c06600af574be



   apprentissage v2 : 5899 découpes | validation : 700


/tmp/ipykernel_23/2648117937.py:358: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt.zero_grad(); p.backward(); torch.nn.utils.clip_grad_norm_(m2.parameters(), 20); opt.step(); pertes.append(float(p))


   époque  1 | perte 0.567 | validation : 83.4%
   époque  2 | perte 0.254 | validation : 84.7%
   époque  3 | perte 0.218 | validation : 85.7%
   époque  4 | perte 0.190 | validation : 86.4%
   époque  5 | perte 0.176 | validation : 86.9%
   époque  6 | perte 0.158 | validation : 87.1%
   époque  7 | perte 0.152 | validation : 87.1%
   époque  8 | perte 0.139 | validation : 87.9%
   époque  9 | perte 0.127 | validation : 87.7%
   époque 10 | perte 0.122 | validation : 87.4%

   v2 (chiffres seuls) — Minerois Ster
   seuil | segments lus | accord Gemini (même / différent) | n°31 (commun / seul)
    0.90 |           83 |   66 / 3                        | 11 / 1
    0.80 |           97 |   72 / 5                        | 12 / 1
    0.60 |          145 |   77 / 14                       | 12 / 1

   v2 — spa (365 découpes, exclu de l'apprentissage)
   seuil | découpes lues | justes | fausses | justes parmi celles de Gemini
    0.90 |           352 |    352 |       0 | 299/299
    0.80 |   